# T10 — Tools

Ein **Tool** ist eine Funktion, deren Aufruf das LLM anfordern kann. Das LLM führt sie nicht selbst aus: Es gibt eine strukturierte Anfrage zurück, die Anwendung führt die Funktion aus und schickt das Ergebnis zurück.

Diese Reihe baut Schritt für Schritt einen RAG-Agenten auf:

| Notebook | Was dazukommt |
|---|---|
| **T10** | Ein Tool definieren; den Ablauf Anfrage-Ausführung-Antwort von Hand durchspielen |
| T11 | `ToolAgent` übernimmt den Ablauf; mehrere Tools; strukturierte Ausgabe |
| T12 | Retrieval als Tool; Ablauf von Hand |
| T13 | Retrieval-Tool in einem `ToolAgent` |
| T14 | RAG-Agent als Tool eines übergeordneten Agenten (Subagent) |

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

## Ein Tool definieren

Ein Tool hat einen **Namen**, eine **Beschreibung**, die dem LLM sagt, wann es das Tool verwenden soll, ein **JSON Schema** seiner Parameter und eine asynchrone `call`-Methode mit der Logik. `SumTwoNumbers` aus `ki_kompass_rag/tool_agents.py`:

```python
class SumTwoNumbers(Tool):
    name = "sum_two_numbers"
    description = "Sum two numbers."
    parameters = _schema(
        number_1=("number", "The first number."),
        number_2=("number", "The second number."),
    )

    async def call(self, args: dict[str, Any]) -> dict[str, Any]:
        a, b = args.get("number_1"), args.get("number_2")
        if not isinstance(a, int | float) or not isinstance(b, int | float):
            raise ValueError("number_1 and number_2 must be numbers.")
        return {"result": a + b}
```

`json_schema()` ist das, was das LLM sieht. Wirft `call` einen Fehler, schickt `ToolAgent` (T11) die Fehlermeldung an das LLM zurück, das es mit korrigierten Argumenten erneut versuchen kann.

In [ ]:
import json
import sys

from loguru import logger

from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.llms.openai import OpenAILLM

from ki_kompass_rag.tool_agents import SumTwoNumbers

logger.remove()
logger.add(sys.stderr, level="WARNING")

tool_sum = SumTwoNumbers()
print(json.dumps(tool_sum.json_schema(), indent=2))
print(await tool_sum.call({"number_1": 5, "number_2": 10}))

## Der Tool-Call-Ablauf

1. **Anfrage**: Die Nachricht der Nutzerin oder des Nutzers senden; das LLM antwortet mit einem Tool-Call statt mit Text.
2. **Ausführung**: Die angeforderten Tools ausführen.
3. **Antwort**: Die Ergebnisse zurückschicken; das LLM schreibt die endgültige Antwort.

Mit `tool_choice="auto"` darf das LLM auch direkt ohne Tool antworten.

### 1. Anfrage

In [ ]:
user_message = LLMMessage(
    role=Roles.USER,
    content=[MessageContent(type="text", text="Sum the numbers 15 and 27 for me.")],
)
llm = OpenAILLM(tools=[tool_sum], tool_choice="auto")

response = await llm.generate([user_message])

print(f"Content: {response.content[0].text!r}")
for tool_call in response.tool_calls or []:
    print(
        f"Tool call {tool_call.id}: {tool_call.function.name}({tool_call.function.arguments})"
    )

### 2. Ausführung

Die Anwendung parst die Argumente und führt das Tool aus. Die Ergebnisse werden nach der ID des Tool-Calls abgelegt, weil das LLM dasselbe Tool in einer Antwort mehrmals anfordern kann.

In [ ]:
tools = {tool.name: tool for tool in llm.tools or []}
results = {}
for tool_call in response.tool_calls or []:
    args = json.loads(tool_call.function.arguments)
    results[tool_call.id] = await tools[tool_call.function.name].call(args)

print(results)

### 3. Antwort

Jedes Ergebnis geht als `TOOL`-Nachricht zurück, über `tool_call_id` verknüpft, nach der Nachricht des Assistenten, die es angefordert hat.

In [ ]:
tool_messages = [
    LLMMessage(
        role=Roles.TOOL,
        name=tool_call.function.name,
        tool_call_id=tool_call.id,
        content=[MessageContent(type="text", text=json.dumps(results[tool_call.id]))],
    )
    for tool_call in response.tool_calls or []
]

final_response = await llm.generate([user_message, response, *tool_messages])
print(final_response.content[0].text)